# Data Contracts — Validation Demo

This notebook demonstrates the `data_contracts` module by running the validation
gate against both **valid** and **invalid** synthetic datasets.

### Contents
1. Setup & imports
2. Helper — synthetic data builder
3. **Reviews contract** — valid data ✓
4. **Reviews contract** — invalid data (multiple failure modes) ✗
5. **Google Trends contract** — valid data ✓
6. **Google Trends contract** — invalid data ✗
7. Combined `run_contracts()` with JSON report
8. `enforce_contracts()` — pipeline abort demo
9. Report inspection

## 1 — Setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure project root is on the path
PROJECT_ROOT = Path(os.getcwd()).parent
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import json
import pandas as pd
import pyarrow as pa
from datetime import datetime, timezone, timedelta

from src.processing.data_contracts import (
    validate_reviews,
    validate_trends,
    run_contracts,
    enforce_contracts,
    ContractViolationError,
    REVIEWS_CONTRACT_DEFAULTS,
    TRENDS_CONTRACT_DEFAULTS,
)

REPORT_PATH = str(PROJECT_ROOT / 'data' / 'processed' / 'quality_report.json')
print('✓ Imports OK')
print(f'  Report will be written to: {REPORT_PATH}')

## 2 — Synthetic data helpers

In [ ]:
def make_reviews_table(
    n: int = 20,
    inject_null_product_ids: int = 0,
    inject_null_ratings: int = 0,
    inject_null_texts: int = 0,
    inject_bad_ratings: list = [],    # list of out-of-range ints to inject
    inject_short_texts: int = 0,      # reviews with < 10 chars
    type_cast_rating_as_string: bool = False,
) -> pa.Table:
    """Build a synthetic Bronze-style reviews PyArrow table."""
    from datetime import datetime, timezone

    base_ts = datetime(2024, 1, 1, tzinfo=timezone.utc)
    products = [f'P{i:04d}' for i in range(1, 6)]   # 5 products → ~4 reviews each

    product_ids  = [products[i % 5] for i in range(n)]
    review_ids   = [f'R{i:06d}' for i in range(n)]
    ratings      = [((i % 5) + 1) for i in range(n)]     # cycles 1-5
    texts        = [f'This product worked really well for me — review number {i}.' for i in range(n)]
    times        = [base_ts + timedelta(days=i) for i in range(n)]

    # Injections
    for idx in range(inject_null_product_ids):
        product_ids[idx] = None
    for idx in range(inject_null_ratings):
        ratings[idx] = None
    for idx in range(inject_null_texts):
        texts[idx] = None
    for i, bad in enumerate(inject_bad_ratings):
        ratings[i] = bad
    for idx in range(inject_short_texts):
        texts[-(idx+1)] = 'meh'

    rating_type = pa.string() if type_cast_rating_as_string else pa.int8()
    raw_ratings = [str(r) if type_cast_rating_as_string else r for r in ratings]

    return pa.table({
        'ProductID':       pa.array(product_ids,  pa.string()),
        'ReviewID':        pa.array(review_ids,   pa.string()),
        'Rating':          pa.array(raw_ratings,  rating_type),
        'ReviewText':      pa.array(texts,         pa.string()),
        'Title':           pa.array([f'Title {i}' for i in range(n)], pa.string()),
        'SubmissionTime':  pa.array(times, pa.timestamp('ms', tz='UTC')),
        'HelpfulCount':    pa.array([i % 10 for i in range(n)], pa.int32()),
        'NotHelpfulCount': pa.array([i % 3  for i in range(n)], pa.int32()),
        'ReviewPhotoCount':pa.array([0]*n, pa.int16()),
        'IsRecommended':   pa.array([True]*n, pa.bool_()),
    })


def make_trends_df(
    keywords: list = None,
    n_weeks:  int  = 52,
    inject_empty_series: list = [],    # keyword names to zero out
    inject_missing_keywords: list = [], # keywords to drop entirely
    start: str = '2023-01-01',
) -> pd.DataFrame:
    """Build a synthetic Google Trends DataFrame."""
    import numpy as np
    kws = keywords or ['hyaluronic', 'niacinamide', 'sunscreen']
    dates = pd.date_range(start=start, periods=n_weeks, freq='W')
    data = {'date': dates}
    for kw in kws:
        if kw in inject_missing_keywords:
            continue  # omit column
        values = [None]*n_weeks if kw in inject_empty_series else list(np.random.randint(10, 100, n_weeks))
        data[kw] = values
    return pd.DataFrame(data)


def show_result(result):
    """Pretty-print a ContractResult."""
    status = '✓ PASSED' if result.passed else '✗ FAILED'
    print(f'\n=== Contract: {result.dataset}  [{status}] ===')
    for c in result.checks:
        icon = '✓' if c.passed else ('✗' if c.critical else '⚠')
        label = '[CRITICAL]' if c.critical and not c.passed else ''
        print(f'  {icon}  {c.name:<40}  {c.message}  {label}')
        if c.detail and not c.passed:
            for k, v in c.detail.items():
                print(f'       {k}: {v}')

print('✓ Helpers defined')

## 3 — Reviews contract: **valid data** ✓

In [ ]:
valid_reviews = make_reviews_table(n=50)
print(f'Table shape: {valid_reviews.num_rows} rows × {valid_reviews.num_columns} columns')
print(f'Schema:\n{valid_reviews.schema}\n')

result_valid = validate_reviews(valid_reviews)
show_result(result_valid)

## 4 — Reviews contract: **invalid data** ✗

We inject multiple failure modes at once to see how each check behaves:

### 4a — Null values in critical fields

In [ ]:
null_reviews = make_reviews_table(n=30, inject_null_product_ids=3, inject_null_ratings=2)
result_nulls = validate_reviews(null_reviews)
show_result(result_nulls)

### 4b — Out-of-range ratings

In [ ]:
bad_rating_reviews = make_reviews_table(n=20, inject_bad_ratings=[0, 6, -1, 99])
result_ratings = validate_reviews(bad_rating_reviews)
show_result(result_ratings)

### 4c — Reviews shorter than N characters (soft warning)

In [ ]:
short_reviews = make_reviews_table(n=30, inject_short_texts=8)
result_short = validate_reviews(short_reviews, config={'min_review_text_chars': 10})
show_result(result_short)

### 4d — Wrong column types (Rating as string instead of int8)

In [ ]:
wrong_type_reviews = make_reviews_table(n=20, type_cast_rating_as_string=True)
result_types = validate_reviews(wrong_type_reviews)
show_result(result_types)

### 4e — Minimum records per product (sparse table)

In [ ]:
# Only 3 rows for 3 products → some products with < min_records
sparse_reviews = make_reviews_table(n=3)
result_sparse = validate_reviews(sparse_reviews, config={'min_records_per_product_warn': 5})
show_result(result_sparse)

### 4f — Empty table (critical failure)

In [ ]:
empty_reviews = make_reviews_table(n=0)
result_empty = validate_reviews(empty_reviews)
show_result(result_empty)

## 5 — Google Trends contract: **valid data** ✓

In [ ]:
valid_trends = make_trends_df(n_weeks=52)
print(valid_trends.head())
print(f'\nShape: {valid_trends.shape}')

result_trends_valid = validate_trends(
    valid_trends,
    config={
        'required_keywords':   ['hyaluronic', 'niacinamide', 'sunscreen'],
        'expected_date_start': '2023-01-01',
        'expected_date_end':   '2023-12-01',
    }
)
show_result(result_trends_valid)

## 6 — Google Trends contract: **invalid data** ✗

### 6a — Missing keyword columns

In [ ]:
missing_kw_trends = make_trends_df(inject_missing_keywords=['niacinamide'])
result_missing_kw = validate_trends(
    missing_kw_trends,
    config={'required_keywords': ['hyaluronic', 'niacinamide', 'sunscreen']}
)
show_result(result_missing_kw)

### 6b — Completely empty series

In [ ]:
empty_series_trends = make_trends_df(inject_empty_series=['sunscreen'])
result_empty_series = validate_trends(
    empty_series_trends,
    config={'required_keywords': ['hyaluronic', 'niacinamide', 'sunscreen']}
)
show_result(result_empty_series)

### 6c — Date range outside expected window

In [ ]:
old_trends = make_trends_df(start='2020-01-01', n_weeks=52)
result_date_range = validate_trends(
    old_trends,
    config={
        'required_keywords':   ['hyaluronic', 'niacinamide', 'sunscreen'],
        'expected_date_start': '2023-01-01',
        'expected_date_end':   '2023-12-31',
    }
)
show_result(result_date_range)

## 7 — Combined `run_contracts()` — exports JSON report

In [ ]:
# Partially-bad reviews (rating range issue) + valid trends
combined_reviews = make_reviews_table(n=40, inject_bad_ratings=[0, 6])
combined_trends  = make_trends_df(n_weeks=52)

report = run_contracts(
    reviews     = combined_reviews,
    trends      = combined_trends,
    reviews_cfg = {'min_review_text_chars': 20},
    trends_cfg  = {'required_keywords': ['hyaluronic', 'niacinamide', 'sunscreen']},
    report_path = REPORT_PATH,
    run_id      = 'demo_run_001',
)

print(f'\nOverall pass: {report.overall_pass}')
print(f'Report written to: {REPORT_PATH}')

## 8 — `enforce_contracts()` — pipeline abort demo

In [ ]:
# This SHOULD raise ContractViolationError because ProductID has nulls
bad_reviews = make_reviews_table(n=30, inject_null_product_ids=5)

try:
    enforce_contracts(
        reviews=bad_reviews,
        report_path=REPORT_PATH,
        run_id='demo_enforce_fail',
    )
    print('No error raised — contracts passed.')
except ContractViolationError as e:
    print(f'\n🚫 Pipeline correctly aborted:\n{e}')

In [ ]:
# This SHOULD pass — clean data
good_reviews = make_reviews_table(n=50)

try:
    report = enforce_contracts(
        reviews=good_reviews,
        report_path=REPORT_PATH,
        run_id='demo_enforce_pass',
    )
    print(f'\n✅ Gate passed — pipeline may continue.  overall_pass={report.overall_pass}')
except ContractViolationError as e:
    print(f'Unexpected failure: {e}')

## 9 — Report inspection

In [ ]:
# Re-run combined report to get the latest one in the file
report = run_contracts(
    reviews     = make_reviews_table(n=50),
    trends      = make_trends_df(n_weeks=52),
    reviews_cfg = {'min_review_text_chars': 10},
    trends_cfg  = {'required_keywords': ['hyaluronic', 'niacinamide', 'sunscreen']},
    report_path = REPORT_PATH,
    run_id      = 'demo_inspect',
)

with open(REPORT_PATH) as f:
    raw = json.load(f)

print(json.dumps(raw, indent=2))

In [ ]:
# Summary table
rows = []
for contract in raw['contracts']:
    for chk in contract['checks']:
        rows.append({
            'dataset':  contract['dataset'],
            'check':    chk['name'],
            'passed':   chk['passed'],
            'critical': chk['critical'],
            'message':  chk['message'],
        })

df_report = pd.DataFrame(rows)
df_report.style.apply(
    lambda col: ['background-color: #d4edda' if v else 'background-color: #f8d7da' for v in col]
    if col.name == 'passed' else ['']*len(col),
    axis=0
)